# PWV02 : Explore statistical uncertainties from the fits, colored by SED type

- author : Sylvie Dagoret-Campagne
- affiliation : IJCLab/IN2P3/CNRS
- creation date : 2025-10-29 (v1), last update of the 2025 version : 2025-11-22
- adapted to the 2026 merged files on 2026-11-13
- kernel : conda_py313 (python 3.13)
- input : merged Spectractor files of `../2026-11-13-SpectractorExtraction-FromButler` (selected in `PWV00_parameters.py`) and the standard star magnitudes `../2025-10-29-TOOLS/data/targets_magnitudes.csv`

**Goal** : same study of the statistical error on PWV as in `PWV02_ExploreHoloPWVStatisticalError.ipynb`, with the targets colored according to their **spectral type** (targets ordered by increasing B-V in the color bar), and :
- the PWV error, the chi2, `PSF_REG`, `TRACE_R`, `MEANFWHM` and the airmass per target on the same figure,
- a fit of the PWV error as a function of the magnitude y,
- the PWV repeatability from the difference between two consecutive PWV measurements of the OG550 filter, for all nights and for the stable periods, with cuts on the time difference and on the azimuth difference.

The per-night figures (PWV with local interpolation, ...) are made in `PWV02_ExploreHoloPWVStatisticalError.ipynb`.

In [ ]:
import os
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.gridspec import GridSpec
import seaborn as sns

warnings.simplefilter('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

plt.rcParams.update({
    'figure.figsize': (16, 8),
    'axes.labelsize': 'xx-large',
    'axes.titlesize': 'xx-large',
    'xtick.labelsize': 'xx-large',
    'ytick.labelsize': 'xx-large',
    'legend.fontsize': 'xx-large',
})
props = dict(boxstyle='round', facecolor='white', alpha=0.1)

print(f'python {python_version()}, numpy {np.__version__}, pandas {pd.__version__}')

## Configuration

In [ ]:
from PWV00_parameters import (
    version_run, atmfilename, tag,
    FLAG_PWVFILTERS, PWV_FILTER_LIST, PWV_FILTEROG550_LIST,
    PWV_METHOD, PWV_COL, PWV_ERR_COL,
    CHI2_COL, D2CCD_COL, TARGET_COL, FILTER_COL, TIME_COL,
    SIGMA_PWV_REPEAT,
    site_lsst, get_astronomical_midnight, getSelectionCut, DumpConfig,
)
DumpConfig()

In [ ]:
NB_TAG = 'PWV02_StatErrors_SedType'
prefix = 'pwv02_staterr_sed'
pathfigs = f'figs_{NB_TAG}'
pathdata = f'data_{NB_TAG}'
for path in (pathfigs, pathdata):
    os.makedirs(path, exist_ok=True)


def save_fig(fig, name, formats=('png', 'pdf')):
    '''Save a figure in the figure directory, with the notebook prefix.'''
    for ext in formats:
        fig.savefig(f'{pathfigs}/{prefix}_{name}.{ext}', bbox_inches='tight')

In [ ]:
PWVMIN, PWVMAX = 0., 18.          # y-range of the PWV plots [mm]

# the collimator was installed on 2023-09-30 : keep only the nights after
FLAG_WITHCOLLIMATOR = True
DATE_WITHCOLLIMATOR = 20230930

TARGETS_MAG_FILE = '../2025-10-29-TOOLS/data/targets_magnitudes.csv'

# the two fit methods : suffix of the columns in the merged files
METHODS = {'ram': 'spectrogram', 'rum': 'spectrum'}
ERR_YLIM = {'ram': 0.08, 'rum': 0.20}      # y-range of the plots of the PWV error [mm]

FILTER_COLORS = {'empty': 'blue',
                 'BG40_65mm_1': 'green',
                 'BG40': 'green',
                 'OG550_65mm_1': 'red',
                 'OG550': 'red',
                 'FELH0600': 'purple',
                 }
# the two filters used in the study of the errors
FILTERS_STUDIED = [('OG550_65mm_1', 'OG550'), ('empty', 'empty')]

# selection of the pairs of consecutive points for the repeatability study
FLAG_SELECT_MAXDTIME = True
DELTAT_REP = 10.                  # maximum time difference [min] (5 min in the 2025 version, but the consecutive OG550
                                  # spectra of 2026 are typically 6-8 min apart : with 5 min only 1 % of the pairs are kept)
FLAG_SELECT_MAXDAZ = True
DELTAT_AZ = 10.                   # maximum azimuth difference [deg]
AZ_COL = 'AZ'                     # azimuth of the telescope [deg], in [-180, 180]

## Read the merged file and apply the selection

In [ ]:
df_spec = pd.DataFrame(np.load(atmfilename, allow_pickle=True))
df_spec['Time'] = pd.to_datetime(df_spec[TIME_COL], utc=True)
df_spec['nightObs'] = df_spec['id'] // 100_000
df_spec = df_spec.sort_values(by='Time')

unknown = set(df_spec[FILTER_COL].unique()) - set(FILTER_COLORS)
assert not unknown, f'add these filters to FILTER_COLORS : {unknown}'

if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec[FILTER_COL].isin(PWV_FILTER_LIST)]
if FLAG_WITHCOLLIMATOR:
    df_spec = df_spec[df_spec['nightObs'] > DATE_WITHCOLLIMATOR]

# quality cuts
df_sel = df_spec[getSelectionCut(df_spec)].reset_index(drop=True)
print(f'{len(df_sel)} / {len(df_spec)} spectra kept after the quality cuts')
df_sel.groupby(FILTER_COL).size()

## Statistical error on PWV
### Standard star magnitudes
The magnitudes and colors of the standard stars in the LSST filters have been computed in `../2025-10-29-TOOLS`.

In [ ]:
df_mag = pd.read_csv(TARGETS_MAG_FILE, index_col=0)
df_mag.index = df_mag.index.str.replace(' ', '')       # 'HD 185975' -> 'HD185975'
df_mag = df_mag[~df_mag.index.duplicated()].sort_values(by='y')

targets_in_data = set(df_sel[TARGET_COL].unique())
missing = sorted(targets_in_data - set(df_mag.index))
if missing:
    print('WARNING : targets without magnitudes (not shown in the plots per target) :', missing)

# targets ordered by increasing y magnitude, and one color per target
targets = [t for t in df_mag.index if t in targets_in_data]
color_map = dict(zip(targets, sns.color_palette('tab20', n_colors=len(targets))))
print(targets)

### Palette with the spectral type
The targets are sorted by increasing B-V, each spectral type has a color of the `jet` color map.

In [ ]:
df_sed = df_mag.sort_values(by='B_V')
unique_types = list(dict.fromkeys(df_sed['Sp_T'].values))      # keep the order of appearance
N_types = len(unique_types)
type_to_idx = {t: i for i, t in enumerate(unique_types)}

cmap = mpl.cm.jet
norm = mpl.colors.Normalize(vmin=-0.5, vmax=N_types - 0.5)

# one color per target (replaces the tab20 palette)
color_map = {t: cmap(norm(type_to_idx[df_mag.loc[t, 'Sp_T']])) for t in targets}


def add_sed_colorbar(fig, axs):
    '''Horizontal color bar of the spectral types below the axes axs.'''
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axs, ticks=range(N_types), location='bottom', shrink=0.6, aspect=60, pad=0.02)
    cbar.ax.set_xticklabels(unique_types, rotation=45, ha='right')
    cbar.set_label('Spectral Type', fontsize=12)
    return cbar


fig, ax = plt.subplots(figsize=(14, 1.2), layout='constrained')
cbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=ax, orientation='horizontal', ticks=range(N_types))
cbar.ax.set_xticklabels(unique_types, rotation=45, ha='right')
cbar.set_label('Spectral Type', fontsize=12)
save_fig(fig, 'sed_colorbar')
plt.show()
print({t: df_mag.loc[t, 'Sp_T'] for t in targets})

## Per target : error, chi2, PSF and airmass
Several metrics on the same figure, the targets are ordered by magnitude y and colored by spectral type.

In [ ]:
def boxplot_per_target(ax, df, ycol, ylim=None, title=''):
    '''Box plot and points of ycol per target, targets ordered by magnitude y.'''
    sns.boxplot(data=df, x=TARGET_COL, y=ycol, hue=TARGET_COL, order=targets, hue_order=targets,
                palette=color_map, legend=False, fliersize=0, ax=ax)
    sns.stripplot(data=df, x=TARGET_COL, y=ycol, order=targets, color='black', alpha=0.4, size=2, ax=ax)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=90)
    if ylim is not None:
        ax.set_ylim(*ylim)


def robust_ylim(series, qhigh=0.99, margin=1.1):
    return (0., series.quantile(qhigh) * margin)


YVARS = ['PWV [mm]_err_ram', 'PWV [mm]_err_rum', CHI2_COL, 'PSF_REG', 'TRACE_R', 'MEANFWHM', 'AIRMASS']


def multi_boxplot(df, title, name):
    '''One row per metric, one box plot per target.'''
    fig, axs = plt.subplots(len(YVARS), 1, figsize=(18, 3.2 * len(YVARS)), sharex=True, layout='constrained')
    for ax, yvar in zip(axs, YVARS):
        if yvar == 'AIRMASS':
            boxplot_per_target(ax, df, yvar, (2.3, 1.0), 'AIRMASS per target')
        else:
            boxplot_per_target(ax, df, yvar, robust_ylim(df[yvar]), f'{yvar} per target')
        ax.set_ylabel(yvar)
        ax.set_xlabel('')
    add_sed_colorbar(fig, axs)
    plt.suptitle(title, fontsize=22)
    save_fig(fig, name)
    plt.show()


multi_boxplot(df_sel, f'Metrics per target, all filters, {tag}', 'multimetrics_vs_target_allfilters')

In [ ]:
for filt, filtname in FILTERS_STUDIED:
    multi_boxplot(df_sel[df_sel[FILTER_COL] == filt], f'Metrics per target, filter {filtname}, {tag}',
                  f'multimetrics_vs_target_{filtname}')

## Relation of the error with magnitude, chi2, airmass and colors, colored by spectral type
For each target : individual measurements (small points) and median (large point).
For the magnitude y, a line `log10(err) = a * y + b` is fitted on the medians of the targets.

In [ ]:
rng_jitter = np.random.default_rng(0)


def plot_by_target(ax, df, ycol, xspec, ylim, jitter=0., annotate=False):
    '''Scatter of ycol for each target.
    xspec is either a column of df_mag (magnitude or color : one x per target, with a horizontal jitter),
    or a column of df (one x per measurement).'''
    per_target_x = xspec in df_mag.columns
    for target in targets:
        sub = df[df[TARGET_COL] == target]
        if len(sub) == 0:
            continue
        y = sub[ycol].values
        if per_target_x:
            xmed = df_mag.loc[target, xspec]
            x = xmed + rng_jitter.normal(0., jitter, len(y))
        else:
            x = sub[xspec].values
            xmed = np.median(x)
        ymed = np.median(y)
        ax.scatter(x, y, color=color_map[target], alpha=0.5, s=60)
        ax.scatter(xmed, ymed, color=color_map[target], s=150, edgecolor='black', zorder=5, label=target)
        if annotate and ymed < 0.9 * ylim:
            ax.text(xmed, ymed + 0.03 * ylim, f'{ymed:.3f}', color=color_map[target], ha='center', va='bottom', fontsize=12)
    ax.set_ylim(0., ylim)
    ax.grid(True, alpha=0.3, linestyle='--')


def fit_err_vs_mag(df, ycol):
    '''Fit log10(median error per target) = a * y + b. Returns (a, b).'''
    mags, errs = [], []
    for target in targets:
        sub = df[df[TARGET_COL] == target]
        if len(sub) > 0 and sub[ycol].median() > 0:
            mags.append(df_mag.loc[target, 'y'])
            errs.append(sub[ycol].median())
    a, b = np.polyfit(mags, np.log10(errs), 1)
    return a, b


fit_results = []


def grid_by_target(xspec, xlabel, name, jitter=0., annotate=False, fit=False):
    '''PWV error versus xspec : one row per filter, one column per method.'''
    fig, axs = plt.subplots(len(FILTERS_STUDIED), len(METHODS), figsize=(18, 6 * len(FILTERS_STUDIED)), layout='constrained')
    for i, (filt, filtname) in enumerate(FILTERS_STUDIED):
        df_f = df_sel[df_sel[FILTER_COL] == filt]
        for j, (method, label) in enumerate(METHODS.items()):
            ax = axs[i, j]
            ycol = f'PWV [mm]_err_{method}'
            plot_by_target(ax, df_f, ycol, xspec, ERR_YLIM[method], jitter, annotate)
            if fit:
                a, b = fit_err_vs_mag(df_f, ycol)
                m_fit = np.linspace(df_mag.loc[targets, 'y'].min(), df_mag.loc[targets, 'y'].max(), 100)
                ax.plot(m_fit, 10**(a * m_fit + b), 'k-', lw=3, label=f'fit : err = 10^({a:.3f} y + {b:.2f})')
                ax.legend(loc='upper left', fontsize=12)
                fit_results.append({'filter': filtname, 'method': method, 'a': a, 'b': b})
            ax.set_xlabel(xlabel)
            ax.set_ylabel('PWV error [mm]')
            ax.set_title(f'filter {filtname}, {label} ({method})', fontsize=14)
    add_sed_colorbar(fig, axs)
    save_fig(fig, name)
    plt.show()


grid_by_target('y', 'Magnitude y (mag)', 'pwverr_vs_mag', jitter=0.02, annotate=True, fit=True)
df_fit = pd.DataFrame(fit_results)
df_fit.to_csv(f'{pathdata}/{prefix}_pwverr_vs_mag_fit_{version_run}.csv', index=False)
df_fit

In [ ]:
grid_by_target(CHI2_COL, 'CHI2_FIT', 'pwverr_vs_chi2fit')

In [ ]:
grid_by_target('AIRMASS', 'airmass', 'pwverr_vs_airmass')

In [ ]:
SedColors = ['u-g', 'g-r', 'r-i', 'i-z', 'z-y']

for method, label in METHODS.items():
    fig, axs = plt.subplots(len(SedColors), len(FILTERS_STUDIED), figsize=(18, 4 * len(SedColors)), layout='constrained')
    for i, sedcolor in enumerate(SedColors):
        for j, (filt, filtname) in enumerate(FILTERS_STUDIED):
            ax = axs[i, j]
            df_f = df_sel[df_sel[FILTER_COL] == filt]
            plot_by_target(ax, df_f, f'PWV [mm]_err_{method}', sedcolor, ERR_YLIM[method], jitter=0.003, annotate=True)
            ax.set_xlabel(f'{sedcolor} (mag)')
            ax.set_ylabel('PWV error [mm]')
            ax.set_title(f'PWV error vs {sedcolor}, filter {filtname}, {label} ({method})', fontsize=14)
    add_sed_colorbar(fig, axs)
    save_fig(fig, f'pwverr_vs_colors_{method}')
    plt.show()

In [ ]:
# chi2 of the fit versus airmass, per target
fig, axs = plt.subplots(1, len(FILTERS_STUDIED), figsize=(18, 6), layout='constrained')
for ax, (filt, filtname) in zip(axs, FILTERS_STUDIED):
    plot_by_target(ax, df_sel[df_sel[FILTER_COL] == filt], CHI2_COL, 'AIRMASS', df_sel[CHI2_COL].max() * 1.05)
    ax.set_xlabel('airmass')
    ax.set_ylabel(CHI2_COL)
    ax.set_title(f'CHI2_FIT vs airmass, filter {filtname}', fontsize=14)
add_sed_colorbar(fig, axs)
save_fig(fig, 'chi2fit_vs_airmass')
plt.show()

## PWV repeatability from consecutive measurements (OG550)
For each night, the difference `PWV(t_n) - PWV(t_n-1)` between two consecutive OG550 measurements is computed. The pairs are selected with :
- a maximum time difference `DELTAT_REP` (minutes),
- a maximum azimuth difference `DELTAT_AZ` (degrees).

The dispersion of the differences is `sqrt(2) * sigma` where `sigma` is the repeatability of one measurement. It is computed for all the nights and for the periods of stability (selected by eye on the 2025 nights). The standard deviation, the robust dispersion from the MAD (`1.4826 * MAD`) and from the IQR (`IQR / 1.349`) are given.

Note : in the 2025 version the MAD was multiplied by `sqrt(pi/2)`, which converts a *mean* absolute deviation ; the factor 1.4826 is the one for the *median* absolute deviation.

In [ ]:
def std_iqr(x):
    '''Robust standard deviation from the inter-quartile range (IQR / 1.349 for a normal distribution).'''
    q25, q75 = np.quantile(x, [0.25, 0.75])
    return (q75 - q25) / 1.349


def sig_mad(x):
    '''Robust standard deviation from the median absolute deviation (1.4826 * MAD for a normal distribution).'''
    return 1.4826 * np.median(np.abs(x - np.median(x)))


def delta_angle_deg(a1, a2):
    '''Angle difference in degrees, brought back to [-180, 180].'''
    return (a1 - a2 + 180.) % 360. - 180.


def get_midnight(night):
    '''Astronomical midnight (UTC timestamp) of the night YYYYMMDD.'''
    day = datetime.strptime(str(night), '%Y%m%d').date()
    ts = pd.Timestamp(get_astronomical_midnight(site_lsst, day))
    return ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')


# periods of stability (hours with respect to midnight), selected by eye on the 2025 nights
dict_select_stablemoments = {
    20250127: {'tstart': 2.1, 'tstop': 4.},
    20250224: {'tstart': -1.9, 'tstop': 0.1},
    20250226: {'tstart': -1., 'tstop': 0.},
    20250303: {'tstart': -1., 'tstop': -0.1},
    20250311: {'tstart': -2.1, 'tstop': -1.1},
    20250317: {'tstart': 1., 'tstop': 5.2},
    20250319: {'tstart': 0.5, 'tstop': 5.2},
    20250320: {'tstart': 0.6, 'tstop': 5.2},
    20250402: {'tstart': -4., 'tstop': -2.2},
    20250407: {'tstart': -1., 'tstop': 0.},
    20250913: {'tstart': 1., 'tstop': 3.},
    20250917: {'tstart': 0.5, 'tstop': 3.},
}


def consecutive_differences(d, method, night):
    '''Differences of PWV between consecutive points of one night (data sorted in time).'''
    y = d[f'PWV [mm]_{method}'].values
    ey = d[f'PWV [mm]_err_{method}'].values
    t = d['Time']
    out = pd.DataFrame({
        'nightObs': night,
        'method': method,
        'dpwv': np.diff(y),
        'edpwv': np.sqrt(ey[1:]**2 + ey[:-1]**2),
        'dt_min': (t.diff().dt.total_seconds().values[1:]) / 60.,
        'daz': np.abs(delta_angle_deg(d[AZ_COL].values[1:], d[AZ_COL].values[:-1])),
        'stable': False,
    })
    if night in dict_select_stablemoments:
        trel = ((t - get_midnight(night)) / pd.Timedelta(hours=1)).values[1:]
        p = dict_select_stablemoments[night]
        out['stable'] = (trel >= p['tstart']) & (trel <= p['tstop'])
    return out


df_og = df_sel[df_sel[FILTER_COL].isin(PWV_FILTEROG550_LIST)]
diffs = []
for night, d in df_og.groupby('nightObs'):
    d = d.sort_values('Time')
    if len(d) < 2:
        continue
    for method in METHODS:
        diffs.append(consecutive_differences(d, method, night))
df_diff = pd.concat(diffs, ignore_index=True)

# selection of the pairs of points
sel = pd.Series(True, index=df_diff.index)
infoselect_txt = ''
if FLAG_SELECT_MAXDTIME:
    sel &= df_diff['dt_min'].abs() <= DELTAT_REP
    infoselect_txt += r'$|\delta t_{rep}| < $' + f'{DELTAT_REP} min \n'
if FLAG_SELECT_MAXDAZ:
    sel &= df_diff['daz'] <= DELTAT_AZ
    infoselect_txt += r'$|\delta az| < $' + f'{DELTAT_AZ} deg'
df_diff['selected'] = sel
df_diff.to_csv(f'{pathdata}/{prefix}_consecutive_differences_{version_run}.csv', index=False)

print(f'{len(df_diff) // len(METHODS)} pairs of consecutive points, {df_diff[df_diff.method == "rum"].selected.sum()} selected')
print(f'stable nights in the data : {sorted(int(n) for n in df_diff[df_diff.stable].nightObs.unique())}')

In [ ]:
def dispersion(x):
    '''Statistics of a set of PWV differences.'''
    return pd.Series({
        'count': len(x),
        'mean': np.mean(x) if len(x) else np.nan,
        'std': np.std(x) if len(x) else np.nan,
        'sig_mad': sig_mad(x) if len(x) else np.nan,
        'sig_iqr': std_iqr(x) if len(x) else np.nan,
    })


rows = {}
for method in METHODS:
    for label, mask in (('all nights', df_diff.selected), ('stable parts', df_diff.selected & df_diff.stable)):
        x = df_diff[(df_diff.method == method) & mask]['dpwv'].values
        stat = dispersion(x)
        rows[(method, label)] = stat
df_rep = pd.DataFrame(rows).T
df_rep.index.names = ['method', 'selection']
# repeatability of one measurement : sigma(difference) / sqrt(2)
for col in ('std', 'sig_mad', 'sig_iqr'):
    df_rep[f'{col}/sqrt2'] = df_rep[col] / np.sqrt(2.)
df_rep.to_csv(f'{pathdata}/{prefix}_pwv_repeatability_{version_run}.csv')
df_rep.round(3)

In [ ]:
fig, axs = plt.subplots(1, len(METHODS), figsize=(7 * len(METHODS), 6), layout='constrained')
for ax, (method, label) in zip(axs, METHODS.items()):
    for sel_label, mask, color, ypos in (('all nights', df_diff.selected, 'b', 0.75), ('stable parts', df_diff.selected & df_diff.stable, 'r', 0.55)):
        x = df_diff[(df_diff.method == method) & mask]['dpwv'].values
        if len(x) < 2:
            continue
        ax.hist(x, bins=100, range=(-1.5, 1.5), density=True, color=color, alpha=0.6, label=sel_label)
        stat = df_rep.loc[(method, sel_label)]
        txt = (r'$\sigma_{RMS} = $' + f"{stat['std']:.2f} mm\n" + r'$\sigma_{MAD} = $' + f"{stat['sig_mad']:.2f} mm\n"
               + r'$\sigma_{IQR} = $' + f"{stat['sig_iqr']:.2f} mm")
        ax.text(0.65, ypos, txt, transform=ax.transAxes, fontsize=14, color=color, va='top',
                bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7))
    ax.text(0.05, 0.95, infoselect_txt, transform=ax.transAxes, fontsize=14, va='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7))
    ax.set_title(f'PWV repeatability in OG550, {label} ({method})')
    ax.set_xlabel(r'$PWV(t_n) - PWV(t_{n-1})$ (mm)')
    ax.legend(loc='upper right')
save_fig(fig, 'pwvrepeat_allnightsandstableparts_og550')
plt.show()